# EDA 3b — Cascade definition sensitivity: should "outflow to same" count?

**Why this notebook exists.**

This is a question about how **cascade churn** is defined, which sits **upstream of the typology** (churn → `Cascade_Dominance` → four-way typology → mechanism leaves → Yee validation). So it belongs here, at the definition stage — *before* EDA 3's typology and long before EDA 12's leaf validation.

**What we test.** Three ways to treat same-decile ("lateral") outflow:

| mode | cascade churn | counter churn | meaning |
|---|---|---|---|
| `strict` (current design) | IW + OP | OW + IP | cascade = *ladder-crossing* displacement only |
| `asym` (literal reading) | IW + OP + O_same | OW + IP | add sideways out-moves to cascade **only** |
| `sym` (mirror) | IW + OP + O_same | OW + IP + I_same | broaden both arms symmetrically |

where `IW`=Inflow_Wealthier, `OP`=Outflow_Poorer, `OW`=Outflow_Wealthier, `IP`=Inflow_Poorer,
`O_same`=outflow to same decile, `I_same`=inflow from same decile.



In [1]:
import numpy as np
import pandas as pd
from pyprojroot import here
ROOT = here()


DATA_DIR   = ROOT / 'outputs' 



FLOWS = DATA_DIR / 'msoa_cascade_features_enriched_20260625.csv'   # Frame A (London-only) flows
PUB   = DATA_DIR / 'eda4_results_for_phase3_20260626.csv'          # published Typ_A_11 / Typ_A_21

df  = pd.read_csv(FLOWS)
pub = pd.read_csv(PUB)[["msoa11cd", "Typ_A_11", "Typ_A_21"]]
df  = df.merge(pub, on="msoa11cd", validate="1:1")
print(f"loaded {len(df)} MSOAs (Frame A, London-only)")

# --- recover the same-decile (lateral) arms by subtraction, and check the identity ---
for yr in ("11", "21"):
    O_same = df[f"Total_Outflow_{yr}"] - df[f"Outflow_Wealthier_{yr}"] - df[f"Outflow_Poorer_{yr}"]
    I_same = df[f"Total_Inflow_{yr}"]  - df[f"Inflow_Wealthier_{yr}"]  - df[f"Inflow_Poorer_{yr}"]
    df[f"Outflow_Same_{yr}"] = O_same
    df[f"Inflow_Same_{yr}"]  = I_same
    assert (O_same >= -0.5).all() and (I_same >= -0.5).all(), "identity broken (negative same-decile flow)"
print("identity Total_Outflow = OW + OP + O_same holds for all rows (both years) \u2713")


loaded 982 MSOAs (Frame A, London-only)
identity Total_Outflow = OW + OP + O_same holds for all rows (both years) ✓


In [2]:
# --- churn + typology under any mode -----------------------------------------
def churns(d, yr, mode):
    IW = d[f"Inflow_Wealthier_{yr}"]; OP = d[f"Outflow_Poorer_{yr}"]
    OW = d[f"Outflow_Wealthier_{yr}"]; IP = d[f"Inflow_Poorer_{yr}"]
    Os = d[f"Outflow_Same_{yr}"];      Is = d[f"Inflow_Same_{yr}"]
    TM = d[f"Total_Migration_{yr}"]
    if mode == "strict":
        casc, cnt = IW + OP,           OW + IP
    elif mode == "asym":                       # add sideways OUT to cascade only
        casc, cnt = IW + OP + Os,      OW + IP
    elif mode == "sym":                        # broaden both arms symmetrically
        casc, cnt = IW + OP + Os,      OW + IP + Is
    else:
        raise ValueError(mode)
    dom = np.where((casc + cnt) > 0, casc / (casc + cnt), np.nan)
    cds = (casc + cnt) / TM                    # "how much turnover engages the ladder"
    return pd.Series(dom, index=d.index), pd.Series(cds, index=d.index)

def typology(dom, cds, cds_thr, dom_hi=0.52, dom_lo=0.48):
    return pd.Series(
        np.where(cds < cds_thr, "Lateral",
        np.where(dom > dom_hi, "Cascade-led",
        np.where(dom < dom_lo, "Counter-led", "Symmetric"))),
        index=dom.index)

MODES = ["strict", "asym", "sym"]
typ = {}
for mode in MODES:
    for yr in ("11", "21"):
        dom, cds = churns(df, yr, mode)
        thr = cds.quantile(0.25)               # P25 lateral threshold, same rule as EDA 3
        typ[(mode, yr)] = typology(dom, cds, thr)


In [3]:
# --- 0. sanity: strict reproduces the PUBLISHED Typ_A exactly ---------------
for yr in ("11", "21"):
    agree = (typ[("strict", yr)] == df[f"Typ_A_{yr}"]).mean()
    n_rep = (typ[("strict", yr)] == "Cascade-led").sum()
    n_pub = (df[f"Typ_A_{yr}"]   == "Cascade-led").sum()
    print(f"20{yr}: strict vs published Typ_A agreement = {agree*100:5.1f}%   "
          f"cascade-led  repro={n_rep}  published={n_pub}")


2011: strict vs published Typ_A agreement = 100.0%   cascade-led  repro=129  published=129
2021: strict vs published Typ_A agreement = 100.0%   cascade-led  repro=76  published=76


In [4]:
# --- 1. MAGNITUDE: is Adam right that 'quite a few' move sideways? ----------
rows = []
for yr in ("11", "21"):
    OP = df[f"Outflow_Poorer_{yr}"].sum()
    Os = df[f"Outflow_Same_{yr}"].sum()
    TO = df[f"Total_Outflow_{yr}"].sum()
    rows.append({"year": f"20{yr}",
                 "to_poorer (OP)": int(OP),
                 "to_same (O_same)": int(Os),
                 "O_same / OP": round(Os / OP, 2),
                 "O_same as % of all outflow": round(100 * Os / TO, 1)})
mag = pd.DataFrame(rows)
print(mag.to_string(index=False))
print("\n\u2192 Same-decile out-moves are ~0.4x the to-poorer arm and ~17% of all out-migration.")


year  to_poorer (OP)  to_same (O_same)  O_same / OP  O_same as % of all outflow
2011          309399            131551         0.43                        17.1
2021          301568            127077         0.42                        16.6

→ Same-decile out-moves are ~0.4x the to-poorer arm and ~17% of all out-migration.


In [5]:
# --- 2. Does it change the typology? counts by mode ------------------------
def counts(series):
    vc = series.value_counts()
    return {k: int(vc.get(k, 0)) for k in ["Cascade-led", "Counter-led", "Symmetric", "Lateral"]}

print("Typology counts (982 London MSOAs)\n")
for yr in ("11", "21"):
    print(f"-- 20{yr} --")
    tab = pd.DataFrame({mode: counts(typ[(mode, yr)]) for mode in MODES}).T
    print(tab.to_string()); print()


Typology counts (982 London MSOAs)

-- 2011 --
        Cascade-led  Counter-led  Symmetric  Lateral
strict          129          339        268      246
asym            363          143        230      246
sym             138          447        397        0

-- 2021 --
        Cascade-led  Counter-led  Symmetric  Lateral
strict           76          416        244      246
asym            279          187        270      246
sym              82          545        355        0



In [6]:
# --- 3. The catch: symmetric broadening COLLAPSES the cross-decile axis -----
# Once same-decile moves count as 'churn', cross-decile share -> ~1.0 for every
# London MSOA (in the London-only frame, cross + same = all internal migration),
# so the Lateral class is erased and the classifier loses its second axis.
for yr in ("11", "21"):
    _, cds_strict = churns(df, yr, "strict")
    _, cds_sym    = churns(df, yr, "sym")
    print(f"20{yr}  cross-decile share      "
          f"STRICT: min={cds_strict.min():.3f} p25={cds_strict.quantile(.25):.3f} med={cds_strict.median():.3f}"
          f"   |   SYM: min={cds_sym.min():.3f} p25={cds_sym.quantile(.25):.3f} med={cds_sym.median():.3f}")
print("\n\u2192 SYM collapses CDS to ~1.0 \u2014 'cross-decile share' can no longer separate lateral from ladder-crossing turnover.")


2011  cross-decile share      STRICT: min=0.437 p25=0.783 med=0.850   |   SYM: min=1.000 p25=1.000 med=1.000
2021  cross-decile share      STRICT: min=0.452 p25=0.792 med=0.855   |   SYM: min=1.000 p25=1.000 med=1.000

→ SYM collapses CDS to ~1.0 — 'cross-decile share' can no longer separate lateral from ladder-crossing turnover.


In [7]:
# --- 4. Stability of the cascade-led set (retention & provenance) ----------
print("Cascade-led membership vs the strict baseline\n")
for yr in ("11", "21"):
    base = set(df.index[typ[("strict", yr)] == "Cascade-led"])
    for mode in ("asym", "sym"):
        new = set(df.index[typ[(mode, yr)] == "Cascade-led"])
        ret, lost, add = len(base & new), len(base - new), len(new - base)
        pct = 100 * ret / len(base)
        print(f"  20{yr} {mode:4s}: {len(base):3d} \u2192 {len(new):3d}   "
              f"retained {ret:3d} ({pct:4.0f}%)  lost {lost:2d}  added {add:3d}")
    # where do the symmetric additions come from / losses go to?
    sym = typ[("sym", yr)]; strict = typ[("strict", yr)]
    added_idx = df.index[(sym == "Cascade-led") & (strict != "Cascade-led")]
    lost_idx  = df.index[(sym != "Cascade-led") & (strict == "Cascade-led")]
    print(f"        sym additions were previously: {strict.loc[added_idx].value_counts().to_dict()}")
    print(f"        sym losses became:             {sym.loc[lost_idx].value_counts().to_dict()}\n")


Cascade-led membership vs the strict baseline

  2011 asym: 129 → 363   retained 124 (  96%)  lost  5  added 239
  2011 sym : 129 → 138   retained 104 (  81%)  lost 25  added  34
        sym additions were previously: {'Lateral': 19, 'Symmetric': 15}
        sym losses became:             {'Symmetric': 24, 'Counter-led': 1}

  2021 asym:  76 → 279   retained  72 (  95%)  lost  4  added 207
  2021 sym :  76 →  82   retained  59 (  78%)  lost 17  added  23
        sym additions were previously: {'Lateral': 12, 'Symmetric': 11}
        sym losses became:             {'Symmetric': 17}



In [8]:
# --- 5. tidy exports to hand to the supervisor ----------------------------
OUT = DATA_DIR / "eda_3b_cascade_definition_sensitivity_summary.csv"
summary = []
for yr in ("11", "21"):
    for mode in MODES:
        c = counts(typ[(mode, yr)])
        c.update({"year": f"20{yr}", "mode": mode})
        summary.append(c)
summary = pd.DataFrame(summary)[["year", "mode", "Cascade-led", "Counter-led", "Symmetric", "Lateral"]]
summary.to_csv(OUT, index=False)
print(f"wrote {OUT.name}")
print(summary.to_string(index=False))


wrote eda_3b_cascade_definition_sensitivity_summary.csv
year   mode  Cascade-led  Counter-led  Symmetric  Lateral
2011 strict          129          339        268      246
2011   asym          363          143        230      246
2011    sym          138          447        397        0
2021 strict           76          416        244      246
2021   asym          279          187        270      246
2021    sym           82          545        355        0
